# DichVideo Batch OmniVoice from SRT

Upload many `.srt` files and one reference voice audio. This notebook uses a high-quality OmniVoice profile: torch CUDA 12.8, OmniVoice from GitHub, faster-whisper ref transcription, a 20 second mono 24 kHz reference clip, 32 inference steps, and 24-bit WAV output.

Use only with your own voice or with clear permission from the voice owner. Runtime > Change runtime type > GPU before running.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# OmniVoice README recommends torch/torchaudio 2.8.0 CUDA 12.8.
!pip -q install --force-reinstall --no-deps torch==2.8.0+cu128 torchaudio==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U git+https://github.com/k2-fsa/OmniVoice.git faster-whisper soundfile


In [ ]:
# Use models already uploaded to Google Drive. This cell does not download model weights.
# Expected Drive folders:
# - MyDrive/models/OmniVoice
# - MyDrive/models/faster-whisper-large-v3
USE_DRIVE_MODELS = True
OMNIVOICE_MODEL_FOLDER = 'OmniVoice'
ASR_MODEL_FOLDER = 'faster-whisper-large-v3'

if USE_DRIVE_MODELS:
    from google.colab import drive
    from pathlib import Path
    import shutil

    drive.mount('/content/drive')
    drive_models_root = Path('/content/drive/MyDrive/models')
    local_models_root = Path('/content/models')

    def copy_drive_model(folder_name: str, required_file: str | None = None) -> str:
        drive_dir = drive_models_root / folder_name
        local_dir = local_models_root / folder_name
        if required_file and not (drive_dir / required_file).exists():
            raise FileNotFoundError(
                'Model not found in Google Drive. Upload the full model folder so this file exists: '
                f'{drive_dir / required_file}'
            )
        if not required_file and (not drive_dir.exists() or not any(drive_dir.iterdir())):
            raise FileNotFoundError(
                'Model folder not found in Google Drive. Upload the full model folder here: '
                f'{drive_dir}'
            )
        if not local_dir.exists():
            print(f'Copying {drive_dir} -> {local_dir}')
            local_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(drive_dir, local_dir)
        else:
            print(f'Using local model copy: {local_dir}')
        return str(local_dir)

    OMNIVOICE_MODEL_PATH = copy_drive_model(OMNIVOICE_MODEL_FOLDER)
    ASR_MODEL_PATH = copy_drive_model(ASR_MODEL_FOLDER, required_file='model.bin')
else:
    OMNIVOICE_MODEL_PATH = 'k2-fsa/OmniVoice'
    ASR_MODEL_PATH = 'medium'

print('OMNIVOICE_MODEL_PATH =', OMNIVOICE_MODEL_PATH)
print('ASR_MODEL_PATH =', ASR_MODEL_PATH)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

SRT_DIR = Path('/content/dichvideo_srt_uploads')
OUTPUT_DIR = Path('/content/dichvideo_omnivoice_audio')
shutil.rmtree(SRT_DIR, ignore_errors=True)
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
SRT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
REF_AUDIO = None
for name, data in uploaded.items():
    suffix = Path(name).suffix.lower()
    if suffix == '.srt':
        (SRT_DIR / name).write_bytes(data)
    elif suffix in {'.mp3', '.wav', '.m4a', '.flac', '.ogg'}:
        ref_path = Path('/content') / name
        ref_path.write_bytes(data)
        REF_AUDIO = str(ref_path)

if not REF_AUDIO:
    raise RuntimeError('Upload one reference audio file, e.g. audio-truyen.mp3')

print('Reference audio:', REF_AUDIO)
print('SRT files:')
for path in sorted(SRT_DIR.glob('*.srt')):
    print('-', path.name)


In [ ]:
import subprocess, torch
from faster_whisper import WhisperModel

REFERENCE_WAV = '/content/ref.wav'
REFERENCE_START = '0'
REFERENCE_DURATION = '20'
LANGUAGE_ID = 'vi'
ASR_MODEL = ASR_MODEL_PATH

# Best-quality default: use about 20 seconds of clean speech, mono 24 kHz.
# If the source has leading silence, change REFERENCE_START to where speech begins.
subprocess.run([
    'ffmpeg', '-y', '-i', REF_AUDIO,
    '-ss', REFERENCE_START, '-t', REFERENCE_DURATION,
    '-ar', '24000', '-ac', '1',
    REFERENCE_WAV,
], check=True)

print('Reference WAV:', REFERENCE_WAV)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
asr = WhisperModel(ASR_MODEL, device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language=LANGUAGE_ID, vad_filter=True)
REF_TEXT = ' '.join(seg.text.strip() for seg in segments).strip()
print('REF_TEXT =', REF_TEXT)
if not REF_TEXT:
    raise RuntimeError('Whisper did not recognize text from the reference audio. Use a clearer reference clip.')


In [ ]:
# Upload scripts/colab_batch_omnivoice_from_srt.py from this repo.
from google.colab import files
uploaded_worker = files.upload()
WORKER_PATH = '/content/colab_batch_omnivoice_from_srt.py'
for name, data in uploaded_worker.items():
    if name.endswith('.py'):
        open(WORKER_PATH, 'wb').write(data)
        break
print('WORKER_PATH:', WORKER_PATH)


In [ ]:
import torch

MODEL = OMNIVOICE_MODEL_PATH
TIMING_MODE = 'no_cut_sequential'  # no_cut_sequential or fit_segments
MAX_TEMPO = '1.35'
NUM_STEP = '32'  # Best quality/stability; use 16 only when you need faster output.
SPEED = '1.0'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float16' if torch.cuda.is_available() else 'float32'

!python "$WORKER_PATH" \
  --srt-dir "$SRT_DIR" \
  --output-dir "$OUTPUT_DIR" \
  --ref-audio "$REFERENCE_WAV" \
  --ref-text "$REF_TEXT" \
  --reference-start "$REFERENCE_START" \
  --reference-duration "$REFERENCE_DURATION" \
  --model "$MODEL" \
  --device "$DEVICE" \
  --dtype "$DTYPE" \
  --timing-mode "$TIMING_MODE" \
  --max-tempo "$MAX_TEMPO" \
  --speed "$SPEED" \
  --num-step "$NUM_STEP"


In [ ]:
from google.colab import files
files.download('/content/omnivoice_audio_results.zip')
